## Init

In [21]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Parent_Mapping") \
    .getOrCreate()

In [22]:
date = "2026-03-08"
path_to_data = f"gs://wynk-ml-workspace/xstream/parent_mapping/prod/day={date}"
day_df = spark.read.parquet(path_to_data)

26/03/09 09:03:46 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 42 for reason Executor for container container_1764236692086_4767_01_000044 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/09 09:03:46 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 17 for reason Executor for container container_1764236692086_4767_01_000019 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.


In [25]:
#test

from datetime import datetime, timedelta

date = "2026-03-05"
base_path = "gs://data-science-prod-redshift-archive/spectrum/xstream_user_consumption_metrics"
date_format = "%Y-%m-%d"

base_date = datetime.strptime(date, date_format)

paths = [
    f"{base_path}/day={(base_date - timedelta(days=i)).strftime(date_format)}"
    for i in range(30)
]

whole_df = None

for path in paths:
    try:
        temp_df = spark.read.parquet(path)

        if whole_df is None:
            whole_df = temp_df
        else:
            whole_df = whole_df.unionByName(temp_df)

    except Exception:
        print(f"Skipping missing path: {path}")

## Data Exploration

In [5]:
df.show(10, truncate=False)

+----------------------------------------------------------------+---------------------------------------------------------------+
|content_id                                                      |item_id                                                        |
+----------------------------------------------------------------+---------------------------------------------------------------+
|MINITV_EPISODE_amzn1.dv.gti.f2bedbdf-3602-429a-8e44-48392e2b68cc|MINITV_TVSHOW_amzn1.dv.gti.2fcde68d-22be-452e-ae61-60f70a32b090|
|PLAYFLIX_EPISODE_109_20                                         |PLAYFLIX_TVSHOW_ATL0000005006                                  |
|PLAYFLIX_EPISODE_138_18                                         |PLAYFLIX_TVSHOW_ATL0000007087                                  |
|PLAYFLIX_EPISODE_151_34                                         |PLAYFLIX_TVSHOW_ATL0000007432                                  |
|PLAYFLIX_EPISODE_207_21                                         |PLAYFLIX_TVSHOW_A

In [29]:
from pyspark.sql import functions as F
a = day_df.distinct().count()
b = whole_df.distinct().count()

26/03/09 09:07:27 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 82 for reason Executor for container container_1764236692086_4767_01_000094 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/09 09:07:27 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 104 for reason Executor for container container_1764236692086_4767_01_000120 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/09 09:07:27 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 81 for reason Executor for container container_1764236692086_4767_01_000092 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/09 09:07:27 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 110 for reason Executor for container container_1764236692086

In [30]:
print("One day unique content_id count:", a)
print("30 days unique content_id count:", b)

One day unique content_id count: 1190029
30 days unique content_id count: 141326255


checking for duplicates

In [31]:
c = day_df.select("content_id").distinct().count()

In [32]:
print("One day unique content_id count:", a)
print("One day unique content_id count:", c)

One day unique content_id count: 1190029
One day unique content_id count: 1190029


26/03/09 09:11:01 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 149 for reason Executor for container container_1764236692086_4767_01_000168 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/09 09:11:31 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 168 for reason Executor for container container_1764236692086_4767_01_000187 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/09 09:12:01 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 160 for reason Executor for container container_1764236692086_4767_01_000179 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/09 09:12:01 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 189 for reason Executor for container container_17642366920

In [28]:
intersection_count = day_df.select("content_id").intersect(whole_df.select("content_id")).count()
print("Intersection count of content_id between one day and 30 days:", intersection_count)

Intersection count of content_id between one day and 30 days: 113560
